In [1]:
! python --version

Python 3.14.7


In [2]:
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import pickle

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## Initialization

In [3]:
DATA_PATH = Path("../data/House Price Dataset.csv")
TEST_DATA_PATH = Path("../data/Test Data For Prediction.csv")

MODEL_PATH = Path("../models/ridge_house_price_model.pkl")

RANDOM_STATE = 42
TEST_SIZE = 0.20

ALPHAS = [
    0.001,
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0,
]

REFERENCE_YEAR = datetime.now().year

## Load dataset

In [4]:
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")

display(df.head())

Dataset shape: (50, 9)


,id,square_footage,bedrooms,bathrooms,year_built,lot_size,distance_to_city_center,school_rating,price
0,1,1250,2,1.0000,1985,5200,3.2000,7.1000,185000
1,2,1850,3,2.0000,1998,7500,5.6000,8.2000,265000
2,3,1420,3,2.0000,1992,6800,2.8000,6.9000,210000
3,4,2100,4,2.5000,2005,9200,7.3000,8.5000,345000
4,5,1700,3,2.0000,2001,7100,4.1000,7.8000,275000


In [5]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nSummary:")
display(df.describe())

Columns:
['id', 'square_footage', 'bedrooms', 'bathrooms', 'year_built', 'lot_size', 'distance_to_city_center', 'school_rating', 'price']

Data types:


id                           int64
square_footage               int64
bedrooms                     int64
bathrooms                  float64
year_built                   int64
lot_size                     int64
distance_to_city_center    float64
school_rating              float64
price                        int64
dtype: object


Missing values:


id                         0
square_footage             0
bedrooms                   0
bathrooms                  0
year_built                 0
lot_size                   0
distance_to_city_center    0
school_rating              0
price                      0
dtype: int64


Duplicate rows:
0

Summary:


,id,square_footage,bedrooms,bathrooms,year_built,lot_size,distance_to_city_center,school_rating,price
count,50.0000,50.0000,50.0000,50.0000,50.0000,50.0000,50.0000,50.0000,50.0000
mean,25.5000,"1,690.2000",3.0400,1.8500,"1,996.2600","7,232.0000",4.6380,7.7740,"264,600.0000"
std,14.5774,425.1746,0.7548,0.7017,10.1213,"1,892.1093",1.9333,0.8007,"80,336.2830"
min,1.0000,980.0000,2.0000,1.0000,"1,978.0000","4,400.0000",2.1000,6.5000,"160,000.0000"
25%,13.2500,"1,335.0000",2.2500,1.0000,"1,987.2500","5,650.0000",3.0250,7.1000,"195,000.0000"
50%,25.5000,"1,665.0000",3.0000,2.0000,"1,997.5000","7,025.0000",4.0500,7.6500,"245,000.0000"
75%,37.7500,"2,065.0000",4.0000,2.5000,"2,005.0000","9,075.0000",6.6750,8.5000,"345,000.0000"
max,50.0000,"2,400.0000",4.0000,3.0000,"2,012.0000","10,500.0000",8.2000,9.1000,"410,000.0000"


In [6]:
REQUIRED_COLUMNS = {
    "id",
    "square_footage",
    "bedrooms",
    "bathrooms",
    "year_built",
    "lot_size",
    "distance_to_city_center",
    "school_rating",
    "price",
}

missing_columns = REQUIRED_COLUMNS - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

print("All required columns are present.")

All required columns are present.


In [7]:
numeric_columns = [
    "square_footage",
    "bedrooms",
    "bathrooms",
    "year_built",
    "lot_size",
    "distance_to_city_center",
    "school_rating",
    "price",
]

for column in numeric_columns:
    if not pd.api.types.is_numeric_dtype(df[column]):
        raise TypeError(
            f"Column '{column}' must be numeric."
        )

if (df["square_footage"] <= 0).any():
    raise ValueError("square_footage must be greater than zero.")

if (df["bedrooms"] <= 0).any():
    raise ValueError("bedrooms must be greater than zero.")

if (df["bathrooms"] <= 0).any():
    raise ValueError("bathrooms must be greater than zero.")

if (df["lot_size"] <= 0).any():
    raise ValueError("lot_size must be greater than zero.")

if (df["price"] <= 0).any():
    raise ValueError("price must be greater than zero.")

print("Basic validation passed.")

Basic validation passed.


## Feature engineering

In [8]:
def create_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    Create the model features from raw housing data.
    """

    data = data.copy()

    data["property_age"] = (
        REFERENCE_YEAR - data["year_built"]
    )

    features = [
        "square_footage",
        "bedrooms",
        "bathrooms",
        "property_age",
        "lot_size",
        "distance_to_city_center",
        "school_rating",
    ]

    return data[features]

In [9]:
X = create_features(df)
y = df["price"].copy()

print("Features:")
display(X.head())

print("\nTarget:")
display(y.head())

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

Features:


,square_footage,bedrooms,bathrooms,property_age,lot_size,distance_to_city_center,school_rating
0,1250,2,1.0000,41,5200,3.2000,7.1000
1,1850,3,2.0000,28,7500,5.6000,8.2000
2,1420,3,2.0000,34,6800,2.8000,6.9000
3,2100,4,2.5000,21,9200,7.3000,8.5000
4,1700,3,2.0000,25,7100,4.1000,7.8000



Target:


0    185000
1    265000
2    210000
3    345000
4    275000
Name: price, dtype: int64


X shape: (50, 7)
y shape: (50,)


## Train/test split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")

Training samples: 40
Test samples:     10


## Build Ridge pipeline

Ridge regularization penalizes coefficients, and current features have very different scales.

In [11]:
model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "regressor",
        RidgeCV(
            alphas=ALPHAS, # type: ignore
            cv=5,
            scoring="neg_mean_squared_error"
        )
    ),
])

In [12]:
model.fit(X_train, y_train)

ridge = model.named_steps["regressor"]

print("Selected alpha:")
print(ridge.alpha_)

Selected alpha:
0.1


## Test-set prediction

In [13]:
y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    "actual_price": y_test.values,
    "predicted_price": y_pred,
})

comparison["error"] = (
    comparison["predicted_price"]
    - comparison["actual_price"]
)

comparison["absolute_error"] = (
    comparison["error"].abs()
)

display(comparison)

,actual_price,predicted_price,error,absolute_error
0,220000,"222,942.1858","2,942.1858","2,942.1858"
1,190000,"182,888.0808","-7,111.9192","7,111.9192"
2,195000,"193,232.5339","-1,767.4661","1,767.4661"
3,195000,"190,141.8661","-4,858.1339","4,858.1339"
4,170000,"155,118.4353","-14,881.5647","14,881.5647"
5,385000,"381,499.5526","-3,500.4474","3,500.4474"
6,255000,"262,538.3881","7,538.3881","7,538.3881"
7,385000,"379,158.5702","-5,841.4298","5,841.4298"
8,205000,"217,503.7480","12,503.7480","12,503.7480"
9,215000,"224,061.8431","9,061.8431","9,061.8431"


## Calculate final test metrics

In [14]:
test_mae = mean_absolute_error(
    y_test,
    y_pred
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

test_r2 = r2_score(
    y_test,
    y_pred
)

print(f"Test MAE:  ${test_mae:,.2f}")
print(f"Test RMSE: ${test_rmse:,.2f}")
print(f"Test R²:   {test_r2:.4f}")

Test MAE:  $7,000.71
Test RMSE: $8,056.59
Test R²:   0.9884


## Proper 5-fold cross-validation

In [15]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    },
    return_train_score=False,
)

cv_mae = -cv_results["test_mae"]
cv_rmse = -cv_results["test_rmse"]
cv_r2 = cv_results["test_r2"]

print("5-Fold Cross-Validation")
print("-----------------------")

print(
    f"MAE:  ${cv_mae.mean():,.2f} "
    f"+/- ${cv_mae.std():,.2f}"
)

print(
    f"RMSE: ${cv_rmse.mean():,.2f} "
    f"+/- ${cv_rmse.std():,.2f}"
)

print(
    f"R²:   {cv_r2.mean():.4f} "
    f"+/- {cv_r2.std():.4f}"
)

5-Fold Cross-Validation
-----------------------
MAE:  $9,026.61 +/- $3,213.79
RMSE: $12,587.69 +/- $6,195.89
R²:   0.9561 +/- 0.0552


## Compare CV vs final test

In [16]:
metrics_summary = pd.DataFrame({
    "Metric": [
        "MAE",
        "RMSE",
        "R²",
    ],
    "Cross Validation": [
        cv_mae.mean(),
        cv_rmse.mean(),
        cv_r2.mean(),
    ],
    "Test Set": [
        test_mae,
        test_rmse,
        test_r2,
    ],
})

display(metrics_summary)

,Metric,Cross Validation,Test Set
0,MAE,"9,026.6102","7,000.7126"
1,RMSE,"12,587.6871","8,056.5853"
2,R²,0.9561,0.9884


## Extract Ridge coefficients

Because the model contains a scaler, we need to convert the coefficients back into the original feature units.

In [17]:
scaler = model.named_steps["scaler"]
ridge = model.named_steps["regressor"]

scaled_coefficients = ridge.coef_

original_coefficients = (
    scaled_coefficients / scaler.scale_
)

original_intercept = (
    ridge.intercept_
    - np.sum(
        scaled_coefficients
        * scaler.mean_
        / scaler.scale_
    )
)

coefficient_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": original_coefficients,
})

display(coefficient_table)

,feature,coefficient
0,square_footage,-81.0255
1,bedrooms,-918.8148
2,bathrooms,"9,384.3278"
3,property_age,-530.9373
4,lot_size,28.6913
5,distance_to_city_center,"16,685.8982"
6,school_rating,"21,370.4443"


In [18]:
coefficient_table = (
    coefficient_table
    .sort_values(
        "coefficient",
        key=lambda s: s.abs(),
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    coefficient_table.style.format({
        "coefficient": "${:,.4f}"
    })
)

,feature,coefficient
0,school_rating,"$21,370.4443"
1,distance_to_city_center,"$16,685.8982"
2,bathrooms,"$9,384.3278"
3,bedrooms,$-918.8148
4,property_age,$-530.9373
5,square_footage,$-81.0255
6,lot_size,$28.6913


## Model information

In [19]:
model_info = {
    "model": {
        "type": "Ridge",
        "alpha": float(ridge.alpha_),
        "features": X.columns.tolist(),
        "intercept": float(original_intercept),
        "coefficients": {
            feature: float(coefficient)
            for feature, coefficient
            in zip(
                X.columns,
                original_coefficients
            )
        },
    },
    "performance": {
        "test": {
            "mae": float(test_mae),
            "rmse": float(test_rmse),
            "r2": float(test_r2),
        },
        "cross_validation": {
            "folds": 5,
            "mae_mean": float(cv_mae.mean()),
            "mae_std": float(cv_mae.std()),
            "rmse_mean": float(cv_rmse.mean()),
            "rmse_std": float(cv_rmse.std()),
            "r2_mean": float(cv_r2.mean()),
            "r2_std": float(cv_r2.std()),
        },
    },
}

model_info

{'model': {'type': 'Ridge',
  'alpha': 0.1,
  'features': ['square_footage',
   'bedrooms',
   'bathrooms',
   'property_age',
   'lot_size',
   'distance_to_city_center',
   'school_rating'],
  'intercept': -48365.53321069607,
  'coefficients': {'square_footage': -81.02546308233569,
   'bedrooms': -918.8148387544409,
   'bathrooms': 9384.327788002447,
   'property_age': -530.9372988776561,
   'lot_size': 28.691331656162237,
   'distance_to_city_center': 16685.898244103282,
   'school_rating': 21370.44434896604}},
 'performance': {'test': {'mae': 7000.712592340246,
   'rmse': 8056.58526901707,
   'r2': 0.9883993447661958},
  'cross_validation': {'folds': 5,
   'mae_mean': 9026.610157092213,
   'mae_std': 3213.790617793189,
   'rmse_mean': 12587.687120978719,
   'rmse_std': 6195.889441672849,
   'r2_mean': 0.9560812296443704,
   'r2_std': 0.05520155994281308}}}

## Test a single prediction

In [20]:
sample_property = pd.DataFrame([{
    "square_footage": 1800,
    "bedrooms": 3,
    "bathrooms": 2,
    "property_age": 15,
    "lot_size": 7500,
    "distance_to_city_center": 5.5,
    "school_rating": 8.2,
}])

predicted_price = model.predict(
    sample_property
)[0]

print(
    f"Predicted price: ${predicted_price:,.2f}"
)

Predicted price: $296,031.86


## Batch predictions

In [21]:
test_df = pd.read_csv(TEST_DATA_PATH)
sample_properties = create_features(test_df)

predictions = model.predict(
    sample_properties
)

result = sample_properties.copy()

result["predicted_price"] = predictions

display(result)

,square_footage,bedrooms,bathrooms,property_age,lot_size,distance_to_city_center,school_rating,predicted_price
0,1550,3,2.0000,29,6800,4.1000,7.6000,"252,588.6435"
1,2200,4,2.5000,18,9600,7.0000,8.8000,"363,905.1186"
2,1180,2,1.0000,44,5100,2.5000,6.7000,"171,432.3915"
3,1870,3,2.5000,25,7900,5.5000,8.3000,"303,356.4418"
4,1430,3,1.5000,36,6200,3.3000,7.2000,"214,791.2788"
5,2300,4,3.0000,13,10300,8.0000,9.2000,"408,467.4308"
6,1650,3,2.0000,27,7100,4.2000,7.8000,"260,098.0500"
7,1020,2,1.0000,47,4400,2.0000,6.5000,"150,102.6836"
8,2100,4,2.5000,21,9200,6.8000,8.6000,"351,327.0519"
9,1350,3,1.0000,38,5700,3.1000,7.0000,"193,562.3430"


## Final summary

In [22]:
print("=" * 60)
print("HOUSE PRICE RIDGE MODEL")
print("=" * 60)

print(f"Model: Ridge")
print(f"Selected alpha: {ridge.alpha_}")

print("\nFeatures:")
for feature in X.columns:
    print(f"  - {feature}")

print("\nCross-validation performance:")
print(f"  MAE:  ${cv_mae.mean():,.2f}")
print(f"  RMSE: ${cv_rmse.mean():,.2f}")
print(f"  R²:   {cv_r2.mean():.4f}")

print("\nFinal test performance:")
print(f"  MAE:  ${test_mae:,.2f}")
print(f"  RMSE: ${test_rmse:,.2f}")
print(f"  R²:   {test_r2:.4f}")

print("\nModel artifacts:")
print(f"  {MODEL_PATH}")
# print(f"  {METADATA_PATH}")

HOUSE PRICE RIDGE MODEL
Model: Ridge
Selected alpha: 0.1

Features:
  - square_footage
  - bedrooms
  - bathrooms
  - property_age
  - lot_size
  - distance_to_city_center
  - school_rating

Cross-validation performance:
  MAE:  $9,026.61
  RMSE: $12,587.69
  R²:   0.9561

Final test performance:
  MAE:  $7,000.71
  RMSE: $8,056.59
  R²:   0.9884

Model artifacts:
  ../models/ridge_house_price_model.pkl


## Save the trained model

In [ ]:
pickle.dump(model, open(MODEL_PATH, 'wb'))

print(f"Model saved to: {MODEL_PATH.resolve()}")

Model saved to: C:\Users\DELL\Documents\CV\2026\Jobs\HSBC\Test 2\property-value-system\models\ridge_house_price_model.pkl


In [ ]:
loaded_model = pickle.load(open(MODEL_PATH, 'rb'))

loaded_prediction = loaded_model.predict(
    sample_property
)[0]

print(f"Original prediction: ${predicted_price:,.2f}")

print(f"Loaded prediction:   ${loaded_prediction:,.2f}")

assert np.isclose(
    predicted_price,
    loaded_prediction
)

print("Model save/load verification passed.")

Original prediction: $296,031.86
Loaded prediction:   $296,031.86
Model save/load verification passed.


## Save model metadata

In [ ]:
import json

METADATA_PATH = Path("../models/ridge_house_price_model_info.json")

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        model_info,
        f,
        indent=2,
    )

print(
    f"Metadata saved to: "
    f"{METADATA_PATH.resolve()}"
)

Metadata saved to: C:\Users\DELL\Documents\CV\2026\Jobs\HSBC\Test 2\property-value-system\models\ridge_house_price_model_info.json


In [26]:
print("=" * 60)
print("HOUSE PRICE RIDGE MODEL")
print("=" * 60)

print(f"Model: Ridge")
print(f"Selected alpha: {ridge.alpha_}")

print("\nFeatures:")
for feature in X.columns:
    print(f"  - {feature}")

print("\nCross-validation performance:")
print(f"  MAE:  ${cv_mae.mean():,.2f}")
print(f"  RMSE: ${cv_rmse.mean():,.2f}")
print(f"  R²:   {cv_r2.mean():.4f}")

print("\nFinal test performance:")
print(f"  MAE:  ${test_mae:,.2f}")
print(f"  RMSE: ${test_rmse:,.2f}")
print(f"  R²:   {test_r2:.4f}")

print("\nModel artifacts:")
print(f"  {MODEL_PATH}")
print(f"  {METADATA_PATH}")

HOUSE PRICE RIDGE MODEL
Model: Ridge
Selected alpha: 0.1

Features:
  - square_footage
  - bedrooms
  - bathrooms
  - property_age
  - lot_size
  - distance_to_city_center
  - school_rating

Cross-validation performance:
  MAE:  $9,026.61
  RMSE: $12,587.69
  R²:   0.9561

Final test performance:
  MAE:  $7,000.71
  RMSE: $8,056.59
  R²:   0.9884

Model artifacts:
  ..\models\ridge_house_price_model.pkl
  ..\models\ridge_house_price_model_info.json
